# [실습] 전처리 파이프라인과 데이터 누수 — 점수를 믿어도 되는가

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# Chapter 4 실습. 이 점수를 믿어도 되는가?

**필수 머신러닝 · Chapter 4**

| Part | 주제 |
| --- | --- |
| Part 1 | 너무 좋은 점수 의심하기 |
| Part 2 | 전처리를 올바른 순서로 |
| Part 3 | 점수를 믿는 법: 교차검증과 시간 순 분할 |
| Part 4 | 보고할 숫자 정하기 |

## 분석 의뢰서

> **의뢰 부서**: 은행 마케팅팀 (Chapter 3의 후속 의뢰)
> **목적**: 지난번 모델을 다음 캠페인에 실제로 쓰려 한다. 그런데 팀원 한 명이 통화 시간 변수를 추가했더니 점수가 훨씬 좋아졌다고 한다.
> **제약**: 모델이 약속한 가입 수가 실제와 크게 다르면 캠페인 예산 승인을 받을 수 없다.
> **질문**: 어느 모델의 점수를 믿고 보고해야 하는가? 그 점수가 실제 캠페인에서도 나온다고 어떻게 확신하는가?

## 이 실습의 질문

> **통화 시간을 뺀 모델과 넣은 모델, 무작위 분할과 시간 순 분할 — 마케팅팀에 "다음 캠페인에서 기대할 가입 수"로 보고할 숫자는 어느 것인가? 그리고 그 숫자를 믿을 수 있다고 어떻게 설득할 것인가?**

이 실습에는 새 모델이 나오지 않습니다. Chapter 3의 모델을 그대로 쓰면서, 그 점수를 **의심하고, 고쳐 만들고, 다시 재는** 연습을 합니다.

## 이 노트북을 쓰는 법

| 표시 | 뜻 |
| --- | --- |
| **[직접 작성]** | 빈 코드 셀을 직접 채웁니다. 강의 교안의 코드를 보지 않고 먼저 써 보세요 |
| **[판단]** | 코드가 아니라 생각을 적는 칸입니다. 셀을 더블클릭해 `나의 답:` 뒤에 적으세요 |
| **[제공]** | 미리 채워 둔 코드입니다. 읽고 실행하세요 |
| **힌트** | 막혔을 때만 펼치세요 |
| **먼저 예상하기** | 결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다 |

## 준비

Chapter 3과 같은 `bank-full.csv`(통화 45,211건)를 이 노트북과 같은 폴더에 두세요. 이번에는 Chapter 3에서 뺐던 통화 시간(`duration`) 열을 **지우지 않고** 둡니다.

In [25]:
# [제공] 이 실습에서 쓰는 라이브러리
import platform
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

# 그래프의 한글 글꼴 (환경에 맞는 글꼴이 없으면 글자가 깨질 수 있습니다)
if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

---
# Part 1. 너무 좋은 점수 의심하기

## 1-1. 준비: Chapter 3의 데이터 다시 만들기

**[제공]** Chapter 3에서 한 준비를 한 번에 합니다. 데이터를 불러와 `target`을 만들고, 같은 방법(80 : 20, `random_state=42`, `stratify`)으로 나눕니다. 이번에는 `duration` 열을 지우지 않습니다.

| 이름 | 뜻 |
| --- | --- |
| `num_cols` | 숫자 피처 4개: `age`, `balance`, `campaign`, `previous` |
| `cat_cols` | 범주 피처 8개: `poutcome`, `housing`, `loan`, `contact`, `month`, `job`, `marital`, `education` |
| `features` | 두 목록을 합친 Chapter 3 모델의 피처 12개 |

In [26]:
# [제공] 데이터 준비 (Chapter 3과 같음, duration은 남겨 둠)
df = pd.read_csv("/Users/limsunghun/python/DI/필수 머신러닝/bank-full.csv", sep=";")
df["target"] = (df["y"] == "yes").astype(int)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["target"])

num_cols = ["age", "balance", "campaign", "previous"]
cat_cols = ["poutcome", "housing", "loan", "contact", "month", "job", "marital", "education"]
features = num_cols + cat_cols
y_train, y_test = train_df["target"], test_df["target"]
print(len(train_df), len(test_df), "| 평가용 가입률", round(y_test.mean(), 3))

36168 9043 | 평가용 가입률 0.117


### 실습 도구 (실행만 하세요)

| 도구 | 하는 일 |
| --- | --- |
| `record(이름, 정답, 확률, 분할)` | 점수를 결과표에 남긴다: ROC AUC, PR AUC, 평가 데이터의 가입률, 상위 1,000명 중 가입 수, 무작위로 1,000명을 골랐을 때의 기대 가입 수, 배수 |
| `show_log()` | 결과표를 보여 준다 |

In [27]:
# [제공] 실습 도구
results = []

def record(name, y_true, prob, split="무작위"):
    y_true = np.asarray(y_true); prob = np.asarray(prob)
    top = y_true[np.argsort(-prob, kind="stable")][:1000].sum()
    base = y_true.mean()
    row = {"실험": name, "분할": split,
           "ROC AUC": round(roc_auc_score(y_true, prob), 3),
           "PR AUC": round(average_precision_score(y_true, prob), 3),
           "평가 가입률": round(base, 3),
           "상위 1,000명 가입": int(top),
           "무작위 1,000명 기대": round(1000 * base),
           "배수": round(top / (1000 * base), 1)}
    results.append(row)
    print(f"기록: [{name} / {split}] ROC AUC {row['ROC AUC']:.3f} | 상위 1,000명 가입 {row['상위 1,000명 가입']}명 (무작위의 {row['배수']}배)")

def show_log():
    return pd.DataFrame(results)

## 1-2. Chapter 3의 모델 다시 만들기

**[직접 작성]** Chapter 3과 같은 방법으로 로지스틱 회귀를 만들고, 평가용 고객의 가입 확률을 `prob_base`에 저장한 뒤 `record("Chapter 3 모델", y_test, prob_base)`로 기록하세요.

1. `pd.get_dummies(df[features])`로 원핫 인코딩 → 학습용·평가용 행으로 나누기 (`.loc[train_df.index]`)
2. `StandardScaler`를 **학습용으로** `fit`하고, 학습용·평가용을 `transform`
3. `LogisticRegression(max_iter=3000)`으로 학습하고 `predict_proba(...)[:, 1]`

<details><summary>힌트</summary>

```python
X = pd.get_dummies(df[features]).astype(float)
X_tr, X_te = X.loc[train_df.index], X.loc[test_df.index]
scaler = StandardScaler().fit(X_tr)
model = LogisticRegression(max_iter=3000).fit(scaler.transform(X_tr), y_train)
prob_base = model.predict_proba(scaler.transform(X_te))[:, 1]
```
</details>

In [28]:
# 여기에 코드를 작성하세요

X = pd.get_dummies(df[features]).astype(float)
X_tr, X_te = X.loc[train_df.index], X.loc[test_df.index]

scaler = StandardScaler().fit(X_tr)
model = LogisticRegression(max_iter=3000).fit(scaler.transform(X_tr), y_train)
prob_base = model.predict_proba(scaler.transform(X_te))[:, 1]

record("Chapter 3 모델", y_test, prob_base)

기록: [Chapter 3 모델 / 무작위] ROC AUC 0.773 | 상위 1,000명 가입 464명 (무작위의 4.0배)


ROC AUC가 0.9까지 오를거같다.

결과가 Chapter 3과 같은지(ROC AUC 0.773 안팎) 확인하세요.

## 1-3. 통화 시간을 넣어 보면

### 먼저 예상하기

결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다. Chapter 3 실습에서 피처를 더하고 설정을 바꿨을 때 ROC AUC가 얼마나 움직였는지 떠올려 보세요.

> **[판단]** 예상: 통화 시간(`duration`, 초)을 피처에 하나 더하면 ROC AUC가 0.773에서 얼마가 될까요?
>
> 나의 답:

**[직접 작성]** 1-2의 코드에서 피처 목록을 `features + ["duration"]`으로 바꿔 같은 모델을 만들고, 확률을 `prob_dur`에 저장한 뒤 `record("통화 시간 추가", y_test, prob_dur)`로 기록하세요.

In [29]:
# 여기에 코드를 작성하세요

features_dur = features + ["duration"]

X_d = pd.get_dummies(df[features_dur]).astype(float)
X_d_tr, X_d_te = X_d.loc[train_df.index], X_d.loc[test_df.index]

scaler_d = StandardScaler().fit(X_d_tr)
model_d = LogisticRegression(max_iter=3000).fit(scaler_d.transform(X_d_tr), y_train)
prob_dur = model_d.predict_proba(scaler_d.transform(X_d_te))[:, 1]

record("통화 시간 추가", y_test, prob_dur)

기록: [통화 시간 추가 / 무작위] ROC AUC 0.906 | 상위 1,000명 가입 578명 (무작위의 4.9배)


ROC AUC가 0.906까지 올랐으며, 내가 예측한 것과 같다. 

먼저 의심해야한다. 피처 하나로 성능이 이렇게 뛰면, 그 피처가 정답과 너무 가깝지 않은지부터 확인해야 한다.

> **[판단]** ROC AUC가 얼마나 올랐나요? 예상과 비교해 보세요. Chapter 3 실습에서 다른 피처를 더했을 때의 변화와 견주면, 이 변화는 어느 정도로 큰가요? 이 결과를 보고 먼저 기뻐해야 할까요, 먼저 의심해야 할까요?
>
> 나의 답:

## 1-4. 이 변수는 어디서 오는가

**[직접 작성]** **학습용 데이터**에서 다음 두 가지를 확인하세요.

1. 가입(`target` = 1)과 미가입(0) 고객의 통화 시간 하위 25%, 중앙값, 상위 25%
2. 통화 시간이 60초 미만인 통화와 600초(10분) 초과인 통화의 가입률과 통화 수

<details><summary>힌트</summary>

1번은 `train_df.groupby("target")["duration"].quantile([0.25, 0.5, 0.75]).unstack()`.
2번은 조건을 만들어 `train_df.loc[조건, "target"].agg(["mean", "size"])`.
</details>

In [30]:
# 여기에 코드를 작성하세요

# 1. 가입/미가입별 통화 시간 분위수
q = train_df.groupby("target")["duration"].quantile([0.25, 0.5, 0.75]).unstack()
q.columns = ["하위 25%", "중앙값", "상위 25%"]
print(q)

# 2. 짧은 통화 vs 긴 통화의 가입률과 통화 수
short = train_df["duration"] < 60
long_ = train_df["duration"] > 600

print("60초 미만:", train_df.loc[short, "target"].agg(["mean", "size"]).round(3).to_dict())
print("600초 초과:", train_df.loc[long_, "target"].agg(["mean", "size"]).round(3).to_dict())

        하위 25%    중앙값  상위 25%
target                       
0         95.0  164.0   278.0
1        245.0  426.0   734.5
60초 미만: {'mean': 0.002, 'size': 3727.0}
600초 초과: {'mean': 0.486, 'size': 3043.0}


분포를 비교한다. 가입 고객의 하위 25%가 미가입 고객의 상위 25%와 거의 같다. 가입 고객의 중앙 값은 미가입 고객 대부분보다 길다. 두 집단의 통화 시간이 거의 겹치지 않는다.

방향은 주로 가입할 고객이라서 통화가 길어진다. 관심있는 고객은 질문하고 조건을 확인하고, 가입하면 본인 확인과 절차 안내까지 이어지니 통화가 길어진다. 관심 없는 고객은 1분 안에 끊다, 상담원의 설득이 일부 영향을 줄 수 있다.

[판단] 위 열 중 예측 시점(전화를 걸기 전)에 알 수 없는 열과 애매한 열을 고르고, 각각 이유를 적으세요. (힌트: campaign의 "마지막 통화 포함"을 눈여겨보세요.)

나의 답:

알 수 없는 열은 duration이다

통화가 끝나야 생기는 값이다. 전하를 걸기전에는 존재하지않는다.

애매한 열은 campaign이다. 

이값은 캠페인 이 끝난 뒤 집계한 최종 통화 수다. 보통 가입하면 더 ㅣ이상 전화하지 않고, 가입하지 않으면 계속 전화를 건다. 그래서 최종 통화 수는 고객이 언제 가입 했는가? 라는 결과가 들어가야한다. 

> **[판단]** 통화 시간과 가입 사이의 관계는 어느 방향일까요? 통화가 길어서 가입한 것일까요, 가입할 고객이라서 통화가 길어진 것일까요? 마케팅팀이 "누구에게 전화할까"를 정하는 시점에 이 값을 알 수 있나요?
>
> 나의 답:

## 1-5. 피처마다 예측 시점 점검하기

데이터 설명서에 적힌 각 열의 뜻입니다. 마케팅팀은 **전화를 걸기 전에** 누구에게 걸지 정해야 합니다.

| 열 | 설명서의 뜻 |
| --- | --- |
| `age`, `job`, `marital`, `education` | 고객 정보 |
| `balance`, `housing`, `loan` | 계좌 잔액, 주택 대출, 개인 대출 |
| `contact` | 연락 수단 |
| `month`, `day` | 마지막 통화의 월, 일 |
| `campaign` | 이번 캠페인에서 이 고객에게 건 통화 수 (**마지막 통화 포함**) |
| `pdays`, `previous`, `poutcome` | 이전 캠페인 이후 경과일, 이전 캠페인 통화 수, 이전 캠페인 결과 |
| `duration` | 마지막 통화의 시간 (초) |

> **[판단]** 위 열 중 예측 시점(전화를 걸기 전)에 **알 수 없는** 열과 **애매한** 열을 고르고, 각각 이유를 적으세요. (힌트: `campaign`의 "마지막 통화 포함"을 눈여겨보세요.)
>
> 나의 답:

---
# Part 2. 전처리를 올바른 순서로

Part 1의 누수는 **변수**에 숨어 있었습니다. 이 Part의 누수는 **코드의 순서**에 숨어 있습니다. 이제부터는 통화 시간을 쓰지 않고, Chapter 3 모델의 피처 12개(`features`)만 씁니다.

## 2-1. 스케일러의 기준을 전체 데이터로 잡으면

**[직접 작성]** 1-2와 같은 모델을 만들되, 스케일러를 **전체 데이터(학습용 + 평가용)**로 `fit`한 버전의 확률을 `prob_leak`에 저장하세요. 그다음 `prob_base`와 `prob_leak`의 ROC AUC를 소수점 넷째 자리까지 출력해 비교하세요. (이 실험은 결과표에 기록하지 않습니다.)

<details><summary>힌트</summary>

`StandardScaler().fit(X)`처럼 나누기 전의 `X` 전체로 `fit`하면 됩니다. 나머지는 1-2와 같습니다. 출력은 `print(round(roc_auc_score(y_test, prob_leak), 4))`.
</details>

In [31]:
# 여기에 코드를 작성하세요

# 스케일러를 전체 데이터(학습용 + 평가용)로 fit
scaler_all = StandardScaler().fit(X)
model_leak = LogisticRegression(max_iter=3000).fit(scaler_all.transform(X_tr), y_train)
prob_leak = model_leak.predict_proba(scaler_all.transform(X_te))[:, 1]

print("학습용으로 fit:", round(roc_auc_score(y_test, prob_base), 4))
print("전체로 fit    :", round(roc_auc_score(y_test, prob_leak), 4))

학습용으로 fit: 0.7729
전체로 fit    : 0.7729


In [32]:
compare = pd.DataFrame({
    "학습용 평균": scaler.mean_,
    "전체 평균": scaler_all.mean_,
    "학습용 표준편차": scaler.scale_,
    "전체 표준편차": scaler_all.scale_,
}, index=X.columns)

print(compare.loc[num_cols].round(3))

            학습용 평균     전체 평균  학습용 표준편차   전체 표준편차
age         40.893    40.936    10.627    10.619
balance   1365.493  1362.272  3068.501  3044.732
campaign     2.764     2.764     3.104     3.098
previous     0.582     0.580     2.409     2.303


전체 데이터 fit 과 학스용 데이터 fit이 점수가 같게 나왔다.

규칙을 지켜야하는 이유는 평가의 전제가 깨진다. 평가용 데이터는 "모델이 아직 보지 못한 미래 고객"을 흉내 낸다. 실제 배포 때는 미래 고객의 평균과 분산을 미리 알 수 없다.가용으로 기준을 잡는 순간 실제로는 불가능한 정보를 쓴 셈이 된다. 이번에 점수가 같았던 건 운이 좋았던 것이지 방법이 옳았기 때문이 아니다.

> **[판단]** 두 점수는 얼마나 다른가요? 왜 이런 결과가 나왔을까요? 점수에 차이가 없다면, 그래도 "기준은 학습용에서만" 규칙을 지켜야 하는 이유는 무엇일까요?
>
> 나의 답:

## 2-2. 기준에 타깃이 들어가면: 타깃 인코딩

**타깃 인코딩**은 범주를 "그 범주의 가입률"로 바꾸는 인코딩입니다. 여기서는 통화한 날(월 + 일, 예: `may-12`)을 범주로 써서, 날마다 가입률을 계산해 피처 하나로 만듭니다.

**[제공]** 날짜 범주를 만듭니다.

In [33]:
# [제공] 통화한 날(월-일) 범주
day_key = df["month"] + "-" + df["day"].astype(str)
key_tr, key_te = day_key.loc[train_df.index], day_key.loc[test_df.index]
print("날짜 범주 수:", day_key.nunique(), "| 날짜당 통화 수 중앙값:", int(day_key.value_counts().median()))

날짜 범주 수: 318 | 날짜당 통화 수 중앙값: 29


**[직접 작성]** 날짜별 가입률을 두 가지 방법으로 계산해, 각각 **피처 하나짜리** 로지스틱 회귀를 만들고 평가용 ROC AUC를 비교하세요.

- **올바른 방법**: 가입률을 **학습용 데이터로만** 계산한다 (`train_df["target"].groupby(key_tr).mean()`)
- **누수가 있는 방법**: 가입률을 **전체 데이터로** 계산한다 (`df["target"].groupby(day_key).mean()`)

어느 방법이든 계산한 가입률을 학습용과 평가용 날짜에 `map`으로 붙여 피처로 씁니다. 학습용에 없던 날짜는 학습용 전체 가입률로 채웁니다.

<details><summary>힌트</summary>

```python
rate = train_df["target"].groupby(key_tr).mean()
f_tr = key_tr.map(rate).fillna(y_train.mean()).to_frame()
f_te = key_te.map(rate).fillna(y_train.mean()).to_frame()
m = LogisticRegression().fit(f_tr, y_train)
print(roc_auc_score(y_test, m.predict_proba(f_te)[:, 1]))
```
누수가 있는 방법은 `rate`만 전체 데이터로 바꿉니다.
</details>

In [34]:
# 여기에 코드를 작성하세요

# 날짜 키 만들기 (예: "may-15")
day_key = df["month"] + "-" + df["day"].astype(str)
key_tr = day_key.loc[train_df.index]
key_te = day_key.loc[test_df.index]

def rate_auc(rate):
    f_tr = key_tr.map(rate).fillna(y_train.mean()).to_frame()
    f_te = key_te.map(rate).fillna(y_train.mean()).to_frame()
    m = LogisticRegression().fit(f_tr, y_train)
    return round(roc_auc_score(y_test, m.predict_proba(f_te)[:, 1]), 4)

# 올바른 방법: 학습용으로만 가입률 계산
rate_ok = train_df["target"].groupby(key_tr).mean()

# 누수가 있는 방법: 전체(학습용 + 평가용)로 가입률 계산
rate_leak = df["target"].groupby(day_key).mean()

print("학습용으로 계산:", rate_auc(rate_ok))
print("전체로 계산    :", rate_auc(rate_leak))

학습용으로 계산: 0.7287
전체로 계산    : 0.7481


전체와 학습용에 차이가0.0194차이가 난다,

통화가 몇 건뿐인 날은 누수가 더 커진다. 그날 통화가 n건이면 평가용 고객 한명의 정답은 가입률에 1/n만큼 반영된다. 하루 1000건이면 영향이 0.1%라  거의 티가 나지 않는다.

> **[판단]** 이번에는 두 방법의 점수가 다른가요? 2-1의 스케일러와 무엇이 다르기에 이번에는 차이가 났을까요? "그날 통화가 몇 건뿐인 날"이라면 누수가 더 커질까요, 작아질까요?
>
> 나의 답:

## 2-3. `get_dummies`의 함정

운영에서는 하루치 고객이 조금씩 들어옵니다. **[제공]** 셀은 평가용 고객 중 20명만 뽑아 `get_dummies`로 바꾸고, 학습할 때의 열과 비교합니다.

In [35]:
# [제공] 새 고객 20명이 들어왔다면
new_customers = test_df.sample(20, random_state=1)
train_cols = pd.get_dummies(df[features]).columns
new_cols = pd.get_dummies(new_customers[features]).columns
print("학습할 때 열 수:", len(train_cols), "| 새 고객 20명의 열 수:", len(new_cols))
print("새 고객에게 없는 열 (예시 10개):", sorted(set(train_cols) - set(new_cols))[:10])

학습할 때 열 수: 46 | 새 고객 20명의 열 수: 38
새 고객에게 없는 열 (예시 10개): ['job_entrepreneur', 'job_self-employed', 'job_student', 'job_unknown', 'month_aug', 'month_jan', 'month_sep', 'poutcome_success']


오류가 난다. 스케일러와 모델은 46개 열을 기준으로 학습한다. 38개 열이 들어오면 transform이나 preict_proba에서 “46개 열을 기대했는데 38개가 들어왔다”는 ValueError가 난다. DataFrame으로 넣으면 열 이름이 학습 때와 다르다는 경고나 오류도 함께 난다.

> **[판단]** 열 수가 다르면 학습한 모델에 새 고객을 넣을 때 어떤 일이 생길까요? 지금까지 Chapter 1~3에서 이 문제를 겪지 않은 이유는 무엇일까요?
>
> 나의 답:

## 2-4. 파이프라인으로 다시 만들기

**[직접 작성]** `ColumnTransformer`와 `Pipeline`으로 Chapter 3 모델을 다시 만드세요.

1. `preprocess`: 숫자 열(`num_cols`)에는 `StandardScaler()`, 범주 열(`cat_cols`)에는 `OneHotEncoder(handle_unknown="ignore")`
2. `pipe_model`: `Pipeline([("pre", preprocess), ("clf", LogisticRegression(max_iter=3000))])`
3. **학습용 데이터 그대로**(`train_df[features]`) `fit`하고, 평가용 확률을 `prob_pipe`에 저장해 `record("파이프라인", y_test, prob_pipe)`로 기록
4. 2-3의 `new_customers`에도 `predict_proba`가 동작하는지 확인

<details><summary>힌트</summary>

```python
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
pipe_model = Pipeline([("pre", preprocess), ("clf", LogisticRegression(max_iter=3000))])
pipe_model.fit(train_df[features], y_train)
```
</details>

In [36]:
# 여기에 코드를 작성하세요

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
pipe_model = Pipeline([("pre", preprocess), ("clf", LogisticRegression(max_iter=3000))])
pipe_model.fit(train_df[features], y_train)

# 3. 평가용 확률 저장 및 기록
prob_pipe = pipe_model.predict_proba(test_df[features])[:, 1]
record("파이프라인", y_test, prob_pipe)

# 4. 새 고객에게도 동작하는지 확인
prob_new = pipe_model.predict_proba(new_customers[features])[:, 1]
print("새 고객 예측 수:", len(prob_new))
print(prob_new[:5].round(3))

기록: [파이프라인 / 무작위] ROC AUC 0.773 | 상위 1,000명 가입 465명 (무작위의 4.0배)
새 고객 예측 수: 20
[0.22  0.18  0.117 0.076 0.578]


> **[판단]** 파이프라인의 점수는 1-2의 "손으로 순서 맞추기"와 같은가요? 점수가 같다면, 파이프라인으로 바꿔서 얻은 것은 무엇인가요? 2-1, 2-3과 연결해 적어 보세요.
>
> 나의 답:

기준이 학습용에서만 만들어진다. 2-1에서는 스케일러를 전체 데이터로 fit해도 점수가 0.7729로 같았다. 하지만 그건 우연에 의한 경우일 가능성이 있다. 

새 고객에게 그대로 쓸 수 있다. 2-3에서는 새 고객 20명을 get_dummies로 인코딩 하자 열이 38개만 생겨 학습 때의 46개와 맞지 않았다. 파이프라인은 학습용에서 본 범주 목록을 인코더가 기억해 두고 항상 같은 열을 만든다.

---
# Part 3. 점수를 믿는 법: 교차검증과 시간 순 분할

## 3-1. 5겹 교차검증

Chapter 1~3에서는 평가용 점수를 보며 설정(깊이, $\alpha$, 임계값)을 골랐습니다. 이제는 설정을 고르는 데 **교차검증**을 쓰고, 평가용은 마지막 확인에만 씁니다.

**[직접 작성]** 2-4의 `pipe_model`을 **학습용 데이터로** 5겹 교차검증하세요. 점수 다섯 개와 평균, 표준편차를 출력합니다.

<details><summary>힌트</summary>

```python
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe_model, train_df[features], y_train, cv=cv, scoring="roc_auc")
```
</details>

In [49]:
# 여기에 코드를 작성하세요

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe_model, train_df[features], y_train, cv=cv, scoring="roc_auc")

print("각 겹 점수:", scores.round(4))
print("평균:", scores.mean().round(4))
print("표준편차:", scores.std().round(4))

각 겹 점수: [0.7647 0.7645 0.7595 0.7687 0.762 ]
평균: 0.7639
표준편차: 0.0031


> **[판단]** 교차검증 평균은 평가용 ROC AUC(0.773)와 얼마나 가깝나요? 교차검증에는 평가용 데이터가 쓰였나요? 파이프라인을 통째로 넣었다는 점은 왜 중요할까요?
>
> 나의 답:

ROC AUC와 교차검증 평균에 차이는 약 0.09차이가 나고 가깝다고 말할 수 있다. 교차검증에서는 평가용 데이터를 쓰지 않고 학습용 데이터만 썻다.
전처리까지 겹마다 새로 학습되기 때문이다. 원-핫인코딩이나 스케일링을 교차검증 전에 학습용 전체에 한 번 맞춰 두면, 스케일링의 평균,표준편차 같은 정보에 검증 조각의 데이터가 이미 섞여 들어갔다. 그러면 모델이 검증 데이터를 미리 본 상태가 되어 점수가 실제보다 좋게 나올 수 있다. 파이프라인째로 넣으면 각 겹에서 학습 조각에만 전처리를 맞추고, 검증 조각에는 그것을 적용한다. 

## 3-2. 두 모델을 비교할 때: 먼저 예상하기

결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다. 다음 절에서 세 쌍의 모델을 같은 5겹 교차검증으로 비교합니다.

| 비교 | 모델 A | 모델 B |
| --- | --- | --- |
| ① | 로지스틱 회귀 ($C = 1$, 기본값) | 로지스틱 회귀 ($C = 0.01$, 규제를 강하게) |
| ② | 결정트리 깊이 5 | 결정트리 깊이 8 |
| ③ | 로지스틱 회귀 ($C = 1$) | 결정트리 깊이 8 |

> **[판단]** 예상: 세 비교 중 "B가 A보다 낫다(또는 못하다)"고 자신 있게 보고할 수 있는 비교는 어느 것일까요? 이유도 적어 보세요.
>
> 나의 답:

자신있게 보고 할 수 있는 비교는 3번이다

같은 모델에서 설정만 바꾼 1,2번과 달리 3번은 예측 방식이 완전히 다른 두 모델의 비교이다. 결정트리는 잎 단위로 확률을 내기 때문에 점수가 계단식이고, 깊이 8이면 이분 잎은 표본이 적어 확률이 불안정해진다. ROC AUC는 고객을 확률 순으로 얼마나 잘 줄세우는지를 보는 지표이므로, 매끄러운 확률을 내는 로지스틱 회귀가 더 높을 것으로 예상한다. 차이가 표준편차보다 몇 배 크게 나오고 다섯 겹모두에서 같은 방향이라면 자신 있게 보고 할 수 있다.


## 3-3. fold별 차이로 비교하기

**[제공]** 모델을 쉽게 만드는 함수 `make_pipe`와, 두 모델을 **같은 5겹**으로 평가해 fold별 점수와 차이를 보여 주는 함수 `compare`입니다.

In [50]:
# [제공] 모델 만들기와 대응 비교
def make_pipe(clf, scale=True):
    pre = ColumnTransformer([
        ("num", StandardScaler() if scale else "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])
    return Pipeline([("pre", pre), ("clf", clf)])

def compare(name_a, model_a, name_b, model_b):
    s_a = cross_val_score(model_a, train_df[features], y_train, cv=cv, scoring="roc_auc")
    s_b = cross_val_score(model_b, train_df[features], y_train, cv=cv, scoring="roc_auc")
    diff = s_b - s_a
    print(f"{name_a}: {s_a.round(3)} | 평균 {s_a.mean():.3f} ± {s_a.std():.3f}")
    print(f"{name_b}: {s_b.round(3)} | 평균 {s_b.mean():.3f} ± {s_b.std():.3f}")
    print(f"fold별 차이 (B - A): {diff.round(3)} | 평균 차이 {diff.mean():+.3f}")
    print()

**[직접 작성]** `make_pipe`로 모델 네 개를 만들고, `compare`로 3-2의 세 쌍을 비교하세요. 결정트리는 스케일링이 필요 없으므로 `scale=False`로 만듭니다(교안 2-6). 트리는 `random_state=42`를 줍니다.

<details><summary>힌트</summary>

```python
logit_1 = make_pipe(LogisticRegression(max_iter=3000))
logit_001 = make_pipe(LogisticRegression(max_iter=3000, C=0.01))
tree_5 = make_pipe(DecisionTreeClassifier(max_depth=5, random_state=42), scale=False)
compare("C=1", logit_1, "C=0.01", logit_001)
```
</details>

In [52]:
# 여기에 코드를 작성하세요

logit_1   = make_pipe(LogisticRegression(max_iter=3000))
logit_001 = make_pipe(LogisticRegression(max_iter=3000, C=0.01))
tree_5    = make_pipe(DecisionTreeClassifier(max_depth=5, random_state=42), scale=False)
tree_8    = make_pipe(DecisionTreeClassifier(max_depth=8, random_state=42), scale=False)

compare("C=1", logit_1, "C=0.01", logit_001)        # ①
compare("깊이 5", tree_5, "깊이 8", tree_8)          # ②
compare("로지스틱 C=1", logit_1, "트리 깊이 8", tree_8)  # ③

C=1: [0.765 0.765 0.759 0.769 0.762] | 평균 0.764 ± 0.003
C=0.01: [0.763 0.76  0.759 0.765 0.761] | 평균 0.761 ± 0.002
fold별 차이 (B - A): [-0.002 -0.005 -0.    -0.004 -0.001] | 평균 차이 -0.002

깊이 5: [0.688 0.713 0.7   0.691 0.699] | 평균 0.698 ± 0.009
깊이 8: [0.705 0.715 0.712 0.693 0.711] | 평균 0.707 ± 0.008
fold별 차이 (B - A): [0.016 0.002 0.012 0.002 0.012] | 평균 차이 +0.009

로지스틱 C=1: [0.765 0.765 0.759 0.769 0.762] | 평균 0.764 ± 0.003
트리 깊이 8: [0.705 0.715 0.712 0.693 0.711] | 평균 0.707 ± 0.008
fold별 차이 (B - A): [-0.06  -0.05  -0.047 -0.076 -0.051] | 평균 차이 -0.057



> **[판단]** 비교 ①: fold별 차이의 방향은 일관되나요? 크기는 실무적으로 의미가 있나요? 둘 중 무엇을 고르겠습니까?
>
> 나의 답:

fold별 차이의 방향은 일관된다.

하지만 크기는 실무적으로 의미가 없다.

평균차이는 -0.002로, 겹 사이 표준편차와 같은 수준이다. ROC AUC가 0.764에서 0.761로 버뀌는 것은 고객을 확률 순으로 줄 세우는 결과가 거의 달라지지 않는다. 

> **[판단]** 비교 ②: 다섯 번 모두 같은 쪽이 이겼나요? 차이의 크기는 fold마다 어떻게 다른가요? "깊이 8이 낫다"고 보고하겠습니까?
>
> 나의 답:

다섯 번 모두 깊이 8이 이겼디. 겹별 차이는 모두 양수이고, 깊이 5가 이긴 겹은 하나도 없다.

하지만 크기의 크기는 겹마다 크게 다르다. 1,3,5번 겹에서는 0.012~0.016으로 꽤 벌어졌지만, 2,4번 겹에서는 0.002로 사실상 비긴것과 같다. 어느쪽이 이기는 가는 일관되지만, 얼마나 이기는가는 데이터를 어떻게 나누느냐에 따라 크게 흔들린다.

> **[판단]** 비교 ③: 차이가 표준편차의 몇 배쯤 되나요? 이 비교는 "낫다"고 보고할 수 있나요? 3-2의 예상은 맞았나요?
>
> 나의 답:

차이는 표준편차의 몇 배에서 몇 십매에 이른다. 평균 차이는 -0.057이다. 

3-2의 예상은 대체로 맞았다,

## 3-4. 교차검증으로 설정 고르기

**[직접 작성]** `GridSearchCV`로 로지스틱 회귀의 `C`를 0.01, 0.1, 1, 10 중에서 고르세요. 가장 좋은 설정(`best_params_`)과, 후보별 교차검증 평균·표준편차 표를 출력합니다.

<details><summary>힌트</summary>

파이프라인 안의 설정은 `단계이름__설정이름`으로 가리킵니다.
```python
search = GridSearchCV(make_pipe(LogisticRegression(max_iter=3000)), {"clf__C": [0.01, 0.1, 1, 10]},
                      cv=cv, scoring="roc_auc")
search.fit(train_df[features], y_train)
pd.DataFrame(search.cv_results_)[["param_clf__C", "mean_test_score", "std_test_score"]]
```
</details>

In [58]:
# 여기에 코드를 작성하세요

search = GridSearchCV(
    make_pipe(LogisticRegression(max_iter=3000)),
    {"clf__C": [0.01, 0.1, 1, 10]},
    cv=cv,
    scoring="roc_auc",
)
search.fit(train_df[features], y_train)

print("가장 좋은 설정:", search.best_params_)
print("그때의 교차검증 평균:", round(search.best_score_, 4))

cv_table = pd.DataFrame(search.cv_results_)[["param_clf__C", "mean_test_score", "std_test_score"]]
print(cv_table.round(4))

가장 좋은 설정: {'clf__C': 0.1}
그때의 교차검증 평균: 0.7641
   param_clf__C  mean_test_score  std_test_score
0          0.01           0.7615          0.0021
1          0.10           0.7641          0.0029
2          1.00           0.7639          0.0031
3         10.00           0.7638          0.0031


> **[판단]** `GridSearchCV`가 고른 설정과 다른 후보들의 차이는 3-3의 기준으로 구별할 만한 크기인가요? 여러분이라면 어떤 `C`를 쓰겠습니까?
>
> 나의 답:

구별할 수 없는 크기이다.

GridSearchCV가 고른 C=0.1과 C=1, C=10의 차이는 0.0002~0.0003이다. 각 후보의 표준편차의 1/10수준이므로, 데이터를 다르게 나누면 1등이 얼마든지 바뀔 수 있다.

선택한다면 C=1을 쓰겠다. 성능이 같은 후보들 가운데서는 점수 대신 다른 기준으로 고르는 것이 합리적이다. C=1은 기본값이라 따로 설명할 필요가 ㅇ벗고, 평평한 구간의 가운데에 있어 안정적이다

## 3-5. 시간이 있는 데이터는 시간 순으로

이 파일의 행은 2008년 5월부터 2010년 11월까지 **날짜순**으로 정렬되어 있습니다. **[제공]** 셀은 행을 순서대로 열 묶음으로 나눠 가입률을 보여 줍니다.

In [59]:
# [제공] 시간 순서대로 본 가입률
deciles = pd.Series(np.arange(len(df)) * 10 // len(df), index=df.index)
print(df.groupby(deciles)["target"].mean().round(3).rename("가입률").rename_axis("구간 (0 = 처음 10%)"))

구간 (0 = 처음 10%)
0    0.030
1    0.037
2    0.048
3    0.064
4    0.061
5    0.062
6    0.105
7    0.131
8    0.161
9    0.471
Name: 가입률, dtype: float64


**[직접 작성]** 앞 80%의 행으로 학습하고 뒤 20%의 행으로 평가하는 **시간 순 분할**을 만드세요. 2-4와 같은 파이프라인을 학습용(앞 80%)으로 `fit`하고, 평가용(뒤 20%)의 확률을 `prob_time`에 저장한 뒤 `record("Chapter 3 모델", y_test_time, prob_time, split="시간 순")`으로 기록하세요.

<details><summary>힌트</summary>

```python
cut = int(len(df) * 0.8)
train_time, test_time = df.iloc[:cut], df.iloc[cut:]
y_test_time = test_time["target"]
```
`make_pipe(LogisticRegression(max_iter=3000))`로 새 파이프라인을 만들어 `train_time[features]`로 `fit`합니다.
</details>

In [61]:
# 여기에 코드를 작성하세요

results = []
cut = int(len(df) * 0.8)
train_time, test_time = df.iloc[:cut], df.iloc[cut:]
y_train_time = train_time["target"]
y_test_time = test_time["target"]

time_model = make_pipe(LogisticRegression(max_iter=3000))
time_model.fit(train_time[features], y_train_time)

prob_time = time_model.predict_proba(test_time[features])[:, 1]

record("Chapter 3 모델", y_test_time, prob_time, split="시간 순")

기록: [Chapter 3 모델 / 시간 순] ROC AUC 0.692 | 상위 1,000명 가입 532명 (무작위의 1.7배)


> **[판단]** 시간 순으로 나누면 ROC AUC가 얼마나 달라지나요? 왜 이런 차이가 생길까요? 실제 캠페인과 더 가까운 조건은 어느 쪽인가요?
>
> 나의 답:

0.773에서 0.962로 약0.08 떨어졌다 교차검증 표준편차가 0.003이었으나 그 25배가 넘는 차이가 있었다. 데이터를 나누는 방식만 바궜는데 비교3보다도 큰 차이가 났다. 상위 1000명에게 전화했을 때 기대되는 약316명의 1.7배이다.

> **[판단]** 그런데 시간 순 분할의 "상위 1,000명 가입"은 무작위 분할보다 많거나 적은가요? ROC AUC는 낮아졌는데 이 숫자는 왜 이렇게 나왔을까요? 결과표의 "평가 가입률"과 "배수" 열을 함께 보세요.
>
> 나의 답:

상위 1000명 중 가입자는 시간 순 분할이 무작위 분할보다 많다. ROC AUC는 0.773에서 0.692로 떨어졌는데 , 가입자 수는 오히려 늘어났다.
이유는 평가 데이터의 가입률 자체가 다르기 떄문이다. 무작위 분할의 평가 가입률은 11,7%이고, 시간 순 분할은 31.6%다. 무직위로 1000명에게 전화해도 뭊작위 분할에서는 약117명, 시간 순 분할에서는 약 316명이 가입했다, 시간 순 분할의 평가 시기는 원래 가입하는고개깅 훨씬 많은 시기라서, 모델이 웬만큼만 골라도 가입자수가 많이 나온다.

---
# Part 4. 보고할 숫자 정하기

이제 질문으로 돌아옵니다. **마케팅팀에 "다음 캠페인에서 기대할 가입 수"로 보고할 숫자는 어느 것인가?**

## 4-1. 먼저 예상하기

결과를 보기 전에 예상을 적어 두고, 결과를 본 뒤 비교합니다. 다음 절에서 네 가지 조합의 점수를 한 표에 놓습니다.

| | 무작위 분할 | 시간 순 분할 |
| --- | --- | --- |
| 통화 시간 없음 | 1-2 (기록됨) | 3-5 (기록됨) |
| 통화 시간 있음 | 1-3 (기록됨) | 4-2에서 만듦 |

> **[판단]** 예상: 네 조합 중 ROC AUC가 가장 높은 것과 가장 낮은 것은 무엇일까요? 다음 캠페인에서 실제로 나올 성과에 가장 가까운 것은 어느 칸일까요?
>
> 나의 답:

가정 높은 것은 통화 시간 있음 + 무작위 분할 로 예상한다.

통화 시간은 가입 여부와 매우 강하게 연결된다. 가입하는고객은상담이 길어ㅣ고, 관심이 없는 고객은 금방 전화를 끊기 때문이다. 여기에 미래 시기의 정보까지 학습에 섞이는 무작위 분할이 더해지면, 두가지 유리한 조건이 겹쳐서 점수가 가장 높게 나온다

가장 낮은 것은 통화 시간 없음 + 시간 순 분할로 예상한다.

이미 0.692로 기록이 되어있고 두가지 유리한 조건이 모두 없는 칸이다.


## 4-2. 네 조합 비교하기

**[직접 작성]** 남은 한 칸(통화 시간 있음 + 시간 순 분할)을 만드세요. `features + ["duration"]`을 쓰는 파이프라인을 만들어야 하므로, `make_pipe`의 숫자 열 목록에 `duration`을 더해야 합니다. 확률을 기록하고 `show_log()`로 결과표 전체를 보세요.

<details><summary>힌트</summary>

`make_pipe`는 `num_cols`를 씁니다. 이 셀에서만 직접 만들면 됩니다.
```python
pre_dur = ColumnTransformer([
    ("num", StandardScaler(), num_cols + ["duration"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
```
</details>

In [63]:
# 여기에 코드를 작성하세요


# duration이 있는 원본 데이터 (행 순서는 df와 같음)
df_dur = pd.read_csv("/Users/limsunghun/python/DI/필수 머신러닝/bank-full.csv", sep=";")
df_dur["target"] = (df_dur["y"] == "yes").astype(int)

# 시간 순 분할: 3-5와 같은 지점에서 자름
cut = int(len(df_dur) * 0.8)
train_dur, test_dur = df_dur.iloc[:cut], df_dur.iloc[cut:]

# duration을 숫자 열에 더한 파이프라인
pre_dur = ColumnTransformer([
    ("num", StandardScaler(), num_cols + ["duration"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
dur_time_model = Pipeline([
    ("pre", pre_dur),
    ("clf", LogisticRegression(max_iter=3000)),
])

dur_features = features + ["duration"]
dur_time_model.fit(train_dur[dur_features], train_dur["target"])
prob_dur_time = dur_time_model.predict_proba(test_dur[dur_features])[:, 1]

record("통화 시간 포함", test_dur["target"], prob_dur_time, split="시간 순")

show_log()

기록: [통화 시간 포함 / 시간 순] ROC AUC 0.782 | 상위 1,000명 가입 598명 (무작위의 1.9배)


,실험,분할,ROC AUC,PR AUC,평가 가입률,"상위 1,000명 가입","무작위 1,000명 기대",배수
0,Chapter 3 모델,시간 순,0.692,0.496,0.316,532,316,1.7
1,통화 시간 포함,시간 순,0.782,0.569,0.316,598,316,1.9


> **[판단]** 예상은 맞았나요? 네 조합 중 마케팅팀에 보고할 점수는 어느 칸이고, 왜 그런가요? 통화 시간을 넣은 두 칸은 왜 보고할 수 없나요?
>
> 나의 답:

예상은 대체로 맞았다.

가장 낮은 칸은 예상대로 통화 시간 없음 + 시간 순 분할이다.

보고할 칸은 통화 시간 없음 + 시간순 분할 이다 이 칸만이 실제 캠페인과 같은 두 가지 조건을 갖추고 있다. 첫 번쨰는 전화를 걸기 전에 알 수 있는 정보만 사용했다. 두 번째는 과거 데이터로 학습해서 미래 시기를 예측했다. 

> **[판단]** 마케팅팀은 "1,000통이면 몇 명이 가입하나요?"라고 묻습니다. 시간 순 분할의 "상위 1,000명 가입" 숫자를 그대로 약속해도 될까요? 다음 캠페인의 가입률이 지난 기간과 다를 수 있다는 점을 생각하면, 어떤 형태로 말하는 것이 정직할까요?
>
> 나의 답:

532명을 그대로 약속하면 안 된다. 이 숫자는 평가 시기의 가입률이 31.6%였기 때문에 나온 값이다. 앞에서 본 것처럼 가입률은 시기마다 3%에서 47%까지 크게 달라졌다. 다음 캠페인의 가입률이 10%라면 모델이 같은 실력을 보여도 가입자는 532명보다 훨씬 적을 것이다.


## 4-3. 점검표 채우기

교안 3-6의 여섯 질문을 **보고하려는 모델**(통화 시간 없음, 파이프라인)에 대해 하나씩 답하세요. 각 질문에 "통과 / 미통과 / 확인 못 함"과 근거를 적습니다.

> **[판단]** 점검표:
>
> ① 쓴 변수가 모두 예측 시점에 존재하는가:
>
> ② 전처리 기준을 학습용으로만 잡았는가:
>
> ③ 설정을 고를 때 평가용 데이터를 쓰지 않았는가:
>
> ④ 비교한 차이가 fold 편차보다 크고 의미 있는가:
>
> ⑤ 평가용이 학습용보다 나중인가:
>
> ⑥ 베이스라인과 현행 규칙보다 얼마나 나은가:
>
> 나의 답:

## 4-4. 마케팅팀 보고 메모

분석 결과를 마케팅팀장에게 메시지로 보낸다고 생각하고 쓰세요. 팀장은 머신러닝 용어를 모릅니다. 아래 네 가지가 모두 들어가야 합니다.

1. **보고하는 숫자와 그 근거**: 어떤 조건(분할 방식)에서 잰 숫자인지
2. **통화 시간 모델을 쓰지 않는 이유**: 점수가 더 높은데도 쓰지 않는 이유를 쉬운 말로
3. **1,000통의 기대 효과**: 절대 숫자를 약속할 수 있는지, 아니라면 어떤 형태로 말하는지
4. **주의할 점**: 다음 캠페인의 가입률이 달라질 가능성, 아직 확인하지 못한 것

> **[판단]** 마케팅팀 보고 메모:
>
> 1. 보고하는 숫자와 근거:
>
> 2. 통화 시간 모델을 쓰지 않는 이유:
>
> 3. 1,000통의 기대 효과:
>
> 4. 주의할 점:
>
> 나의 답:

---
# 마무리

## 이 Chapter의 판단 포인트

> **이 정보는 예측 시점에 알 수 있는가? 그리고 두 모델의 점수 차이는 교차검증 편차보다 큰가?**

## 개인 검증 체크리스트에 추가

오늘 실습에서 내가 실제로 겪은 일을 바탕으로, 체크리스트의 줄을 내 말로 적으세요.

예: "점수가 갑자기 좋아진 변수를 보면, 그 값이 언제 기록되는지부터 확인한다."

> **[판단]** 나의 체크리스트 6번(누수를 점검할 때):
>
> 나의 답:

> **[판단]** 나의 체크리스트 7번(점수를 비교하고 보고할 때):
>
> 나의 답:

## 제출물

- 이 노트북 (모든 **[판단]** 칸을 채운 상태)

수고했습니다. 다음 Chapter 5에서는 트리를 여러 개 모아 더 강한 모델을 만들고, 그 모델이 무엇을 보고 예측하는지 SHAP으로 설명합니다.